In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [19]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.lower().endswith('.csv'):
            print(os.path.join(root, f))


/content/drive/MyDrive/Social_Network_Ads.csv


In [20]:
# This is the old incorrect path that caused the error. We store it as a string instead.
csv_path = "/content/drive/MyDrive/SomeFolder/Social_Network_Ads.csv"

In [21]:
import os
import pandas as pd

# Search for the CSV file in Google Drive
target_file = 'Social_Network_Ads.csv'
file_path = None

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if target_file in files:
        file_path = os.path.join(root, target_file)
        break

if file_path:
    print(f"Found file at: {file_path}")
    df = pd.read_csv(file_path)
    print(df.head())
else:
    print(f"Could not find '{target_file}' in your Google Drive. Please double check the file name or upload it directly.")

Found file at: /content/drive/MyDrive/Social_Network_Ads.csv
   Age  EstimatedSalary  Purchased
0   19            19000          0
1   35            20000          0
2   26            43000          0
3   27            57000          0
4   19            76000          0


In [22]:
df = pd.read_csv("/content/drive/MyDrive/Social_Network_Ads.csv")

In [24]:
import os
from google.colab import files

files_to_download = ['comparison_table.csv', 'predictions_set1.csv', 'predictions_set2.csv']

for file in files_to_download:
    if os.path.exists(file):
        files.download(file)
    else:
        print(f"Warning: {file} not found. Ensure you run the code that generates this file first.")

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)
from google.colab import files
import os

sns.set_style("whitegrid")

# ---------------- 1. LOAD DATA ----------------
# Using the path from your screenshot
df = pd.read_csv("/content/drive/MyDrive/Social_Network_Ads.csv")
print("Dataset loaded successfully! Shape:", df.shape)

# Keep only the required columns
df = df[['Age', 'EstimatedSalary', 'Purchased']].copy()

# ---------------- 2. PREPROCESSING ----------------
X = df[['Age', 'EstimatedSalary']].values
y = df['Purchased'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ---------------- 3. TRAIN & EVALUATE 5 MODELS ----------------
configs = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'KNN (k=5)'          : (KNeighborsClassifier(n_neighbors=5), True),
    'SVM (RBF)'          : (SVC(kernel='rbf', probability=True, random_state=42), True),
    'Decision Tree'      : (DecisionTreeClassifier(max_depth=6, random_state=42), False),
    'Random Forest'      : (RandomForestClassifier(n_estimators=200, random_state=42), False),
}

trained, rows = {}, []
for name, (model, needs_sc) in configs.items():
    Xtr, Xte = (X_train_sc, X_test_sc) if needs_sc else (X_train, X_test)
    t0 = time.time(); model.fit(Xtr, y_train); train_t = time.time() - t0
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]
    rows.append({
        'Algorithm'      : name,
        'Accuracy'       : round(accuracy_score(y_test, y_pred), 4),
        'Precision'      : round(precision_score(y_test, y_pred), 4),
        'Recall'         : round(recall_score(y_test, y_pred), 4),
        'F1-Score'       : round(f1_score(y_test, y_pred), 4),
        'ROC-AUC'        : round(roc_auc_score(y_test, y_prob), 4),
        'Train Time (s)' : round(train_t, 4),
    })
    trained[name] = (model, needs_sc)

results = pd.DataFrame(rows).sort_values('Accuracy', ascending=False)
print("\n========= COMPARISON TABLE =========\n")
print(results.to_string(index=False))
results.to_csv('comparison_table.csv', index=False)

# ---------------- 4. SCENARIO PREDICTIONS ----------------
median_salary = df['EstimatedSalary'].median()

def predict_scenarios(pairs, set_name):
    print(f"\n----- Scenario Set: {set_name} -----")
    out = []
    for age, sal in pairs:
        imputed = sal is None
        s = median_salary if imputed else sal
        row = pd.DataFrame([[age, s]], columns=['Age','EstimatedSalary'])
        row_sc = scaler.transform(row)
        preds = {}
        for name,(m,needs_sc) in trained.items():
            x = row_sc if needs_sc else row.values
            preds[name] = int(m.predict(x)[0])
        prob_lr = trained['Logistic Regression'][0].predict_proba(row_sc)[0,1]
        out.append({'Age':age, 'Salary':s, 'Imputed':imputed, 'LR Prob':round(prob_lr,3), **preds})
    out_df = pd.DataFrame(out)
    print(out_df.to_string(index=False))
    return out_df

set1 = [(30, 87000), (40, None), (40, 100000), (50, None)]
set2 = [(18, None), (22, 600000), (35, 2500000), (60, 100000000)]
res1 = predict_scenarios(set1, "Set 1 (realistic)")
res2 = predict_scenarios(set2, "Set 2 (extreme)")
res1.to_csv('predictions_set1.csv', index=False)
res2.to_csv('predictions_set2.csv', index=False)

# ---------------- 5. DOWNLOAD FILES ----------------
print("\n✅ DONE. Preparing downloads...")
files_to_download = ['comparison_table.csv', 'predictions_set1.csv', 'predictions_set2.csv']

for file in files_to_download:
    if os.path.exists(file):
        files.download(file)
    else:
        print(f"Warning: {file} not found.")

Dataset loaded successfully! Shape: (400, 3)

========= COMPARISON TABLE =========

          Algorithm  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Train Time (s)
          KNN (k=5)      0.91     0.8857  0.8611    0.8732   0.9375          0.0070
      Random Forest      0.91     0.8649  0.8889    0.8767   0.9301          1.5107
          SVM (RBF)      0.91     0.8649  0.8889    0.8767   0.9423          0.0255
      Decision Tree      0.87     0.8485  0.7778    0.8116   0.9180          0.0051
Logistic Regression      0.84     0.8571  0.6667    0.7500   0.9102          0.0314

----- Scenario Set: Set 1 (realistic) -----


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but Sta

 Age   Salary  Imputed  LR Prob  Logistic Regression  KNN (k=5)  SVM (RBF)  Decision Tree  Random Forest
  30  87000.0    False    0.118                    0          0          0              0              0
  40  70000.0     True    0.360                    0          0          0              0              0
  40 100000.0    False    0.613                    1          1          1              1              1
  50  70000.0     True    0.810                    1          0          1              0              0

----- Scenario Set: Set 2 (extreme) -----
 Age      Salary  Imputed  LR Prob  Logistic Regression  KNN (k=5)  SVM (RBF)  Decision Tree  Random Forest
  18     70000.0     True    0.007                    0          0          0              0              0
  22    600000.0    False    1.000                    1          1          1              0              1
  35   2500000.0    False    1.000                    1          1          1              1              1


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>